# Ingesta de datos de agricultura en Sonora

Esta libreta de Jupyter se utilizará para la ingesta y análisis de datos relacionados con la agricultura en el estado de Sonora. Para esto utilizaremos dataset disponibles públicamente sobre cultivos, producción y condiciones climáticas en la región a lo largo de los años. Los datos obtenidos serán procesados y transformados para facilitar su análisis y visualización, permitiendo así obtener insights relevantes sobre la agricultura en Sonora.

## Ingesta de datos

### Librerias necesarias

In [57]:
!pip install pydantic


[notice] A new release of pip is available: 25.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
from pathlib import Path
import pandas as pd
from loguru import logger
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pydantic import BaseModel, PositiveInt, ValidationError, Field, ConfigDict, field_validator
from typing import Optional
from tqdm import tqdm

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

RAW_DATA_DIR = Path("../data/raw")


# Cada año usa un identificador de recurso y un nombre de archivo distintos en el portal.
SONORA_AGRICULTURE_URLS = {
    2019: (
        "https://datos.sonora.gob.mx/dataset/3e17a7e8-c8d4-49c4-a426-4dec099cd0cd/"
        "resource/9e455a9c-1733-490e-ba3d-6117ffe4ffe5/download/"
        "agricultura-sonora-2019.xlsx"
    ),
    2020: (
        "https://datos.sonora.gob.mx/dataset/3e17a7e8-c8d4-49c4-a426-4dec099cd0cd/"
        "resource/ff1ec96e-4cc3-4feb-8359-1af90bb19175/download/"
        "agricultura-sonora-2020.xlsx"
    ),
    2021: (
        "https://datos.sonora.gob.mx/dataset/3e17a7e8-c8d4-49c4-a426-4dec099cd0cd/"
        "resource/49a39677-0dbc-49ec-b44b-09030c0fa8f6/download/"
        "agricultura-sonora-2021.xlsx"
    ),
    2022: (
        "https://datos.sonora.gob.mx/dataset/3e17a7e8-c8d4-49c4-a426-4dec099cd0cd/"
        "resource/f815b112-4fd8-4bac-8be5-ae18ff1c2ef8/download/"
        "agricultura_sonora_-2022.xlsx"
    ),
    2023: (
        "https://datos.sonora.gob.mx/dataset/3e17a7e8-c8d4-49c4-a426-4dec099cd0cd/"
        "resource/ed647672-5b4d-4c5f-b2fa-1093a7df799b/download/"
        "cierre-agricola-estatal-mun-ddr_2023.xlsx"
    ),
}





### Descarga de datos

In [ ]:
def download_agriculture_data(start_year: int = 2019, end_year: int = 2023):
    """Descarga los datos de agricultura de Sonora por año."""
    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

    years = range(start_year, end_year + 1)

    for year in tqdm(years, total=len(years)):
        url = SONORA_AGRICULTURE_URLS.get(year)
        dest_file = RAW_DATA_DIR / f"agricultura-sonora-{year}.xlsx"

        if url is None:
            logger.warning(f"Año {year}: No hay URL registrada, se omite la descarga.")
            continue

        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()

            with open(dest_file, "wb") as f:
                f.write(response.content)

            logger.success(f"Año {year}: Descargado con éxito")
        except requests.exceptions.RequestException as e:
            logger.warning(f"Failed to download agriculture {year}: {e}")

    print("Carpeta destino:", RAW_DATA_DIR)
    print("Existe:", RAW_DATA_DIR.exists())
    logger.success("Agriculture download complete.")

In [60]:
download_agriculture_data(start_year=2019, end_year=2023)

 20%|██        | 1/5 [00:00<00:03,  1.05it/s]

2026-09-28 19:40:26.895 | SUCCESS  | __main__:download_agriculture_data:22 - Año 2019: Descargado con éxito


 40%|████      | 2/5 [00:01<00:02,  1.34it/s]

2026-09-28 19:40:27.497 | SUCCESS  | __main__:download_agriculture_data:22 - Año 2020: Descargado con éxito


 60%|██████    | 3/5 [00:01<00:01,  1.65it/s]

2026-09-28 19:40:27.938 | SUCCESS  | __main__:download_agriculture_data:22 - Año 2021: Descargado con éxito


 80%|████████  | 4/5 [00:02<00:00,  1.83it/s]

2026-09-28 19:40:28.388 | SUCCESS  | __main__:download_agriculture_data:22 - Año 2022: Descargado con éxito


100%|██████████| 5/5 [00:02<00:00,  1.67it/s]

2026-09-28 19:40:28.930 | SUCCESS  | __main__:download_agriculture_data:22 - Año 2023: Descargado con éxito
Carpeta destino: ../data/raw
Existe: True
2026-09-28 19:40:28.934 | SUCCESS  | __main__:download_agriculture_data:28 - Agriculture download complete.


In [61]:
for year in range(2019, 2024):
    df = pd.read_excel(RAW_DATA_DIR / f"agricultura-sonora-{year}.xlsx")
    globals()[f"agricultura_sonora_df_{year}"] = df
    logger.info(f"Loaded agricultura-sonora-{year}.xlsx")

2026-09-28 19:40:29.045 | INFO     | __main__:<module>:4 - Loaded agricultura-sonora-2019.xlsx
2026-09-28 19:40:29.099 | INFO     | __main__:<module>:4 - Loaded agricultura-sonora-2020.xlsx
2026-09-28 19:40:29.157 | INFO     | __main__:<module>:4 - Loaded agricultura-sonora-2021.xlsx
2026-09-28 19:40:29.168 | INFO     | __main__:<module>:4 - Loaded agricultura-sonora-2022.xlsx
2026-09-28 19:40:29.234 | INFO     | __main__:<module>:4 - Loaded agricultura-sonora-2023.xlsx


In [62]:
agricultura_sonora_df_2019.head()

,ANO,CIERREYAVAN,CICLO,CDDR,NDDR,CMUN,NMUN,CVECUL,CULTIVO,SUPSEM,SUPCOSE,SUPSINI,PRODTON,RENDMNTO,PMR,VALPROD
0,2019,CIERRE DE PRODUCCION AGRICOLA 2019,3,140,DDR 140 MAGDALENA,64,Trincheras,901,Aceituna,2.0,2.0,0.0,8.2,4.1,13526.82,110.92
1,2019,CIERRE DE PRODUCCION AGRICOLA 2019,3,139,DDR 139 CABORCA,4,Altar,901,Aceituna,0.0,0.0,0.0,0.0,0.0,0.00,0.00
2,2019,CIERRE DE PRODUCCION AGRICOLA 2019,3,139,DDR 139 CABORCA,17,Caborca,901,Aceituna,1211.0,1211.0,0.0,11505.0,9.5,11600.00,133458.00
3,2019,CIERRE DE PRODUCCION AGRICOLA 2019,2,144,DDR 144 HERMOSILLO,30,Hermosillo,1,Acelga,2.0,2.0,0.0,20.0,10.0,10500.00,210.00
4,2019,CIERRE DE PRODUCCION AGRICOLA 2019,1,144,DDR 144 HERMOSILLO,56,San Miguel de Horcasitas,1,Acelga,8.0,8.0,0.0,148.0,18.5,9250.00,1369.00


## Validación de la descarga de datos de agricultura de Sonora

### Schema basandonos en el diccionario de datos de agricultura de Sonora

In [63]:
# class AgriculturaSonoraSchema(BaseModel):
#     """Schema for the Sonora agriculture production dataset."""

#     ANO: Optional[str] = Field(default=None, pattern=r"^\d{4}$")
#     CIERREYAVAN: Optional[str] = None
#     CICLO: Optional[int] = Field(default=None, ge=1, le=3)
#     CDDR: Optional[str] = None
#     NDDR: Optional[str] = None
#     CMUN: Optional[str] = None
#     NMUN: Optional[str] = None
#     CVMES: Optional[str] = None
#     NMES: Optional[str] = None
#     CVECUL: Optional[str] = None
#     CULTIVO: Optional[str] = None
#     CVEVAR: Optional[str] = None
#     DESVAR: Optional[str] = None
#     SUPSEM: Optional[float] = Field(default=None, ge=0)
#     SUPCOSE: Optional[float] = Field(default=None, ge=0)
#     SUPSINI: Optional[float] = Field(default=None, ge=0)
#     PRODTON: Optional[float] = Field(default=None, ge=0)
#     RENDMNTO: Optional[float] = Field(default=None, ge=0)
#     PMR: Optional[float] = Field(default=None, ge=0)
#     VALPROD: Optional[float] = Field(default=None, ge=0)

#     @field_validator(
#         "ANO",
#         "CIERREYAVAN",
#         "CDDR",
#         "NDDR",
#         "CMUN",
#         "NMUN",
#         "CVMES",
#         "NMES",
#         "CVECUL",
#         "CULTIVO",
#         "CVEVAR",
#         "DESVAR",
#         mode="before",
#     )
#     @classmethod
#     def coerce_text_fields(cls, value: Any) -> Optional[str]:
#         if value is None:
#             return None
#         return str(value).strip()


Esta clase ya se encuentra definida en el archivo schemas.py por lo que podemos importar simplemente desde allí.

In [64]:
from sonora_agriclimate_eda.schemas import AgriculturaSonoraSchema

In [65]:
AgriculturaSonoraSchema.model_fields

{'ANO': FieldInfo(annotation=Union[str, NoneType], required=False, default=None, metadata=[_PydanticGeneralMetadata(pattern='^\\d{4}$')]),
 'CIERREYAVAN': FieldInfo(annotation=Union[str, NoneType], required=False, default=None),
 'CICLO': FieldInfo(annotation=Union[int, NoneType], required=False, default=None, metadata=[Ge(ge=1), Le(le=3)]),
 'CDDR': FieldInfo(annotation=Union[str, NoneType], required=False, default=None),
 'NDDR': FieldInfo(annotation=Union[str, NoneType], required=False, default=None),
 'CMUN': FieldInfo(annotation=Union[str, NoneType], required=False, default=None),
 'NMUN': FieldInfo(annotation=Union[str, NoneType], required=False, default=None),
 'CVMES': FieldInfo(annotation=Union[str, NoneType], required=False, default=None),
 'NMES': FieldInfo(annotation=Union[str, NoneType], required=False, default=None),
 'CVECUL': FieldInfo(annotation=Union[str, NoneType], required=False, default=None),
 'CULTIVO': FieldInfo(annotation=Union[str, NoneType], required=False, d

### Función para validar los datos descargados de agricultura de Sonora

In [66]:
def validate_df(df: pd.DataFrame) -> pd.DataFrame:
    validated_rows = []

    for index, row in df.iterrows():
        try:
            agriculture = AgriculturaSonoraSchema(**row.to_dict())
            validated_rows.append(agriculture.model_dump())

        except ValidationError as e:
            print(f"Error en fila {index}:")
            print(e)

    return pd.DataFrame(validated_rows)

In [67]:
df_validated = validate_df(agricultura_sonora_df_2019)